# Ch11 KV Cache 推理优化 教案

**课程名称：** KV Cache 推理优化：让模型生成快起来

**预计总时长：** 50-60 分钟

**源文件：** `Ch11_KV_Cache/Ch11_KV_Cache.ipynb`（共 23 个 Cell，Cell 0-22）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-5 min | 开场 + 环境准备 | Cell 0-3 | 5 min |
| 5-15 min | 问题分析：O(n²) 的计算瓶颈 | Cell 4-5 | 10 min |
| 15-30 min | KV Cache 原理 + 可视化 | Cell 6-10 | 15 min |
| 30-35 min | **休息 + 回顾** | -- | 5 min |
| 35-42 min | 代码实现：带 Cache 的 Attention | Cell 11 | 7 min |
| 42-48 min | 完整 GPT with KV Cache + 性能对比 | Cell 12-16 | 6 min |
| 48-55 min | KV Cache 内存分析 + 真实模型对比 | Cell 17-18 | 7 min |
| 55-60 min | 总结 + 练习 + 下一步 | Cell 19-22 | 5 min |

---

## 课前准备

- [ ] 确认 Python 3.11+ 和 PyTorch 已安装（`import torch; print(torch.__version__)`）
- [ ] 确认 `transformers` 库已安装（Cell 18 需要 `AutoConfig`）
- [ ] 如有 GPU，确认 CUDA 可用：`print(torch.cuda.is_available())`
- [ ] 预跑一遍全部 Cell，确认无报错
- [ ] 确认中文字体可正常显示（Matplotlib 图表含中文标注）
- [ ] 建议学生先复习 Ch5 的自注意力机制（Q/K/V 计算）
- [ ] 准备白板，用于手绘 KV Cache 拼接示意图

---

## 第一段：开场与环境准备（Cell 0-3）

📍 运行 Cell 0-1（Markdown 导读）、Cell 2-3（环境准备代码）

⏱ 时间分配：5 分钟

🎯 本段目标
- 建立学习动机：为什么模型生成越长越慢？
- 预览本章内容：问题分析 -> 原理 -> 实现 -> 性能对比
- 确认环境就绪

🗣 讲课话术

> 大家好！我们在 Ch5 里从零搭建了 GPT 模型，也学会了自回归生成——一个词一个词地往外蹦。但不知道大家有没有想过一个问题：**为什么 ChatGPT 回答长问题时，后面的字会越来越慢？**
>
> 其实这背后有一个 O(n²) 的计算陷阱。今天我们就来解决这个问题。本章的主角叫 **KV Cache**——它是所有 LLM 推理框架（vLLM、TensorRT-LLM、Ollama）的核心优化技术。
>
> 先看 Cell 0 的目录：问题分析、KV Cache 原理、代码实现、性能对比。四个部分，50 分钟搞定。
>
> Cell 1 的学习路线表告诉我们：前置知识是 Ch5 的自注意力机制（Q/K/V 计算），学完本章后下一站是 Ch12 Agent & RAG。
>
> 运行 Cell 3。（运行 Cell 3）看到 `Using device: cuda`。今天的代码很轻量，CPU 也完全能跑。

👀 输出要点
- Cell 3：`Using device: cuda`（或 cpu）

❓ 预判问题
- **Q：KV Cache 和 Flash Attention 是什么关系？**
  A：它们解决不同问题。KV Cache 避免重复计算历史 token 的 K/V，减少计算量。Flash Attention 优化 Attention 的 IO 和内存访问模式，减少显存和提升吞吐。两者可以同时使用。
- **Q：KV Cache 只在推理时用吗？**
  A：对。训练时一次性喂入整个序列，所有位置的 Q/K/V 都要同时算（做并行训练），不需要缓存。KV Cache 是专门为逐 token 生成的推理阶段设计的。

➡️ 转场

> 好，环境就绪。现在让我们先搞清楚问题出在哪——为什么朴素的自回归生成是 O(n²) 的？

---

## 第二段：问题分析——O(n²) 的计算瓶颈（Cell 4-5）

📍 浏览 Cell 4（Markdown 问题分析）、运行 Cell 5（O(n²) vs O(n) 可视化）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解朴素自回归生成为什么是 O(n²)
- 通过具体数字感受计算浪费的严重性
- 引出 KV Cache 的核心思路

🗣 讲课话术

> 先看 Cell 4 的分析。自回归生成是一个词一个词蹦出来的。生成第 1 个词，算 1 次 Attention；生成第 2 个词，要把前 2 个词的 Attention 全算一遍；生成第 n 个词，要算 n 次。
>
> 总计算量是 1 + 2 + 3 + ... + n = **n(n+1)/2**，这就是 O(n²)。
>
> 打个比方：你在考试写作文。写第一个字，你只看一眼题目。写第十个字，你把前面九个字从头读一遍。写第一百个字，你把前面九十九个字全重读一遍。**你每写一个字都要从头读整篇文章**——这效率能不低吗？
>
> 运行 Cell 5 看数字。（运行 Cell 5）
>
> 看左边这张图：红色曲线是朴素方法，像抛物线一样飞速增长；绿色直线是 KV Cache 方法，平稳得多。右边是加速比——序列越长，加速越明显。
>
> 关键数字来了：**当序列长度为 100 时，朴素方法需要 5050 次操作，KV Cache 只需要 100 次，加速 50.5 倍！** 这还只是 100 个 token。想象一下 GPT-4 处理 128K 上下文，没有 KV Cache 几乎不可能实时响应。
>
> 那 KV Cache 怎么做到的？关键洞察是：**之前 token 的 K 和 V 不会变**。你第一步算出来的 K1、V1，第二步、第三步还是一样的值。既然不变，为什么每次都要重新算？**缓存起来就好了！**

👀 输出要点
- Cell 5 图表：左图 O(n²) vs O(n) 对比曲线，右图加速倍数随序列长度线性增长
- Cell 5 文字输出：
  - 朴素方法：5050 次操作
  - KV Cache：100 次操作
  - 加速：50.5x

❓ 预判问题
- **Q：为什么 K 和 V 不会变？**
  A：在 Decoder-only 架构中，由于 Causal Mask 的存在，位置 i 的输出只依赖位置 0 到 i 的输入。已经生成的 token 不会改变，所以它们对应的 K 和 V 也不会变。Q 呢？Q 代表"当前 token 想查询什么"，每一步只需要当前 token 的 Q，之前的 Q 用完就扔了。
- **Q：既然缓存能加速这么多，有什么代价？**
  A：内存！缓存 K 和 V 需要额外的显存。这是经典的时间-空间权衡。稍后我们会算具体需要多少内存。

➡️ 转场

> 问题搞清楚了：朴素方法重复计算太多。解决方案是缓存 K 和 V。接下来我们深入看 KV Cache 到底怎么工作。

---

## 第三段：KV Cache 原理 + 可视化（Cell 6-10）

📍 浏览 Cell 6（关键洞察 Markdown）、Cell 7（内存公式）、Cell 8（深层理论）、运行 Cell 9（Attention 矩阵对比）、运行 Cell 10（工作原理动画）

⏱ 时间分配：15 分钟（理论 8 分钟 + 可视化 7 分钟）

🎯 本段目标
- 理解"只算最后一个 Query"的核心机制
- 掌握 KV Cache 内存公式
- 了解 Prefill 和 Decode 两个阶段
- 通过矩阵和动画可视化建立直觉

🗣 讲课话术

> 先看 Cell 6 的表格。三步生成的过程：
> - Step 1：输入 [A]，计算 Q1、K1、V1，无缓存可用
> - Step 2：输入 [A, B]，只需要计算 Q2、K2、V2，K1 和 V1 从缓存读取
> - Step 3：输入 [A, B, C]，只算 Q3、K3、V3，K1、V1、K2、V2 全从缓存读
>
> 这里有个关键区分——**训练 vs 推理**。训练时一次性喂入整个序列，所有位置的 Q/K/V 都要并行计算，所以不用 Cache。推理时逐 token 生成，只需要当前 token 的 Q，历史 K/V 复用。
>
> 推理还分两个阶段：**Prefill（预填充）** 是首次处理整个 prompt，生成所有位置的 K/V 并缓存；**Decode（解码）** 是逐步生成新 token，每步只算 1 个 Q/K/V 并追加到缓存。
>
> 现在看内存。Cell 7 给出了公式：$\text{KV Memory} = 2 \times B \times n_{kv\_head} \times T \times d_{head} \times \text{dtype\_bytes}$。乘以层数就是总内存。2 是因为 K 和 V 各存一份。
>
> Cell 8 算了两个真实例子，大家感受一下量级：
> - **GPT-3（175B）**：96 层、96 头、seq=2048、FP16。KV Cache 需要 **9.66 GB**！光缓存就快 10 个 G，还没算模型权重。
> - **LLaMA-2-70B（用了 GQA）**：80 层、64 个 Q 头但只有 8 个 KV 头。KV Cache 只需要 **1.34 GB**——GQA 压缩了 8 倍！
>
> 这就是为什么 GQA（Grouped Query Attention）这么重要。多个 Q 头共享一组 K/V，大幅减少缓存内存。
>
> 好，看两张直觉图。运行 Cell 9。（运行 Cell 9）左边蓝色是朴素方法——要算完整的 T×T Attention 矩阵（下三角）。右边绿色是 KV Cache——只算最后一行，1×T。**从一个三角矩阵变成一条线**，计算量差距一目了然。
>
> 运行 Cell 10。（运行 Cell 10）这张大图对比了三步生成过程。上排朴素方法：每一步所有 token 都标红（重复计算），Step 3 需要 9 次矩阵计算。下排 KV Cache：只有新 token 标绿（新计算），旧 token 的 K/V 是蓝色虚线框（缓存读取），每步只需要 3 次矩阵计算。
>
> 底部的文字总结得很好：朴素方法每步重算所有 Q/K/V，KV Cache 只算新 token 的 Q/K/V，旧的直接复用。计算量从 O(n²) 降到 O(n)。

👀 输出要点
- Cell 9：左图 T×T 下三角矩阵 vs 右图 1×T 单行，附文字"生成时只需要最后一个 token 的输出"
- Cell 10：6 子图对比（2 行×3 列），上排红色重复计算 vs 下排绿色+蓝色虚线缓存
- Cell 10 文字：核心差异三行总结

❓ 预判问题
- **Q：为什么 Q 不缓存？**
  A：Q 代表"当前 token 想查询什么"。在生成阶段，我们只关心最新 token 的 Q，之前 token 的 Q 不会再被用到。K 和 V 不同——新 token 的 Q 要和所有历史位置的 K 做点积、对所有历史 V 做加权求和，所以 K 和 V 必须保留完整历史。
- **Q：MQA、GQA、MHA 有什么区别？**
  A：MHA（Multi-Head Attention）每个头独立的 K/V；MQA 所有 Q 头共享一组 K/V，Cache 减少 n_heads 倍但质量略降；GQA 是折中，每 G 个 Q 头共享一组 K/V。LLaMA-2-70B 用 GQA（64 Q 头, 8 KV 组），质量接近 MHA，内存接近 MQA。
- **Q：Prefill 和 Decode 哪个更耗时？**
  A：Prefill 是 compute-bound（要一次性算所有 token 的 K/V），Decode 是 memory-bound（每步只算一个 token 但要读取大量缓存）。对于长 prompt + 短生成的场景，Prefill 可能更耗时；对于短 prompt + 长生成的场景，Decode 的累积时间更长。

➡️ 转场

> 原理搞清楚了，现在动手写代码！我们来看怎么在 Attention 模块里加入 KV Cache。

---

## 休息 + 回顾（第 30-35 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 朴素自回归生成每步重算所有 token 的 K/V，总计算量是 O(n²)——生成 100 个 token 需要 5050 次操作。
2. KV Cache 缓存历史 token 的 K 和 V，每步只算新 token 的 Q/K/V，总计算量降为 O(n)——同样 100 个 token 只需 100 次操作，加速 50.5x。
3. KV Cache 的代价是内存：GPT-3 在 seq=2048 时 KV Cache 需要 9.66 GB，但 GQA 技术（如 LLaMA-2-70B）可以压缩 8 倍到 1.34 GB。

**下一段预告：** 我们要亲手写带 KV Cache 的 Attention 代码，看看 `past_kv` 参数是怎么拼接的。

---

## 第四段：代码实现——带 Cache 的 Attention（Cell 11）

📍 运行 Cell 11（CausalSelfAttentionWithCache 类 + 测试）

⏱ 时间分配：7 分钟

🎯 本段目标
- 理解 forward 函数中 `past_kv` 参数的拼接逻辑
- 理解 `use_cache` 控制缓存的读写
- 验证有缓存和无缓存的输出形状

🗣 讲课话术

> 运行 Cell 11。这是本章最核心的代码——`CausalSelfAttentionWithCache` 类。大家注意和 Ch5 里的普通 Attention 有什么不同。
>
> forward 函数多了两个参数：`past_kv` 和 `use_cache`。逻辑分三步：
>
> **第一步：计算当前的 Q, K, V。** 和之前一样，`self.c_attn(x)` 得到 3C 维向量再 split。
>
> **第二步：如果有缓存，拼接！** 这是关键。代码里写的是：
> ```python
> k = torch.cat([past_k, k], dim=1)  # [B, past_T + T, C]
> v = torch.cat([past_v, v], dim=1)
> ```
> 把之前缓存的 K/V 和新算的 K/V 在序列维度上拼起来。这样新 token 的 Q 就能和所有历史 K 做 Attention 了。
>
> **第三步：返回更新后的缓存。** `present_kv = (k, v)` 包含了完整的历史+新计算的 K/V，供下一步使用。
>
> 还有一个细节：当 T=1（生成阶段只有一个 query）时，不需要 causal mask。因为这个 query 是序列的最后一个位置，它可以看到前面所有位置。
>
> 看测试输出：
> - 无缓存输出形状：`[1, 10, 64]`——10 个 token 进，10 个 token 出
> - 有缓存时新 token 输出：`[1, 1, 64]`——只有 1 个 token 进出！
> - 缓存 K 形状：`[1, 11, 64]`——原来 10 个 + 新 1 个 = 11 个
>
> 大家注意 `[1, 11, 64]` 这个形状。缓存在不断增长，每生成一个 token 就多一个位置。这就是 KV Cache 内存随序列长度线性增长的原因。

👀 输出要点
- 无缓存输出形状：`torch.Size([1, 10, 64])`
- 有缓存时新 token 输出：`torch.Size([1, 1, 64])`
- 缓存 K 形状：`torch.Size([1, 11, 64])`（10 + 1 = 11）

❓ 预判问题
- **Q：代码里那个 TODO 注释是什么意思？**
  A：`# TODO: 注意这里 v 也要拼接，保持 K 和 V 的长度一致`。代码已经写好了 `v = torch.cat([past_v, v], dim=1)`，这个 TODO 是提醒大家：K 拼了，V 也一定要拼，否则 Attention 矩阵和 V 的维度不匹配会报错。
- **Q：为什么 T=1 时不需要 mask？**
  A：Causal mask 的作用是防止位置 i 看到位置 j（j > i）的信息。当 T=1 时只有一个 query，它处于序列的最末位置，前面所有 key 都在它之前，所以不需要遮挡。
- **Q：缓存会无限增长吗？**
  A：是的，标准 KV Cache 随序列长度线性增长。这就是为什么有 block_size 限制（最大上下文长度）。超长序列可以用滑动窗口（Mistral）或 PagedAttention（vLLM）来管理。

➡️ 转场

> 单个 Attention 层搞定了，接下来我们把它堆成完整的 GPT 模型，然后跑性能对比！

---

## 第五段：完整 GPT + 性能对比（Cell 12-16）

📍 运行 Cell 13（GPTWithCache 模型定义）、Cell 15（性能测试）、Cell 16（可视化）

⏱ 时间分配：6 分钟

🎯 本段目标
- 理解多层 KV Cache 的传递机制（每层独立缓存）
- 对比有无 Cache 的实际性能差异
- 体会小模型下加速效果有限，大模型下加速效果显著

🗣 讲课话术

> 运行 Cell 13。`GPTWithCache` 把多个带 Cache 的 Attention 层堆在一起。注意 `past_kv_list` 是一个列表，每层有自己的缓存。另一个关键：position ID 的处理——有缓存时，新 token 的位置要从 `past_length` 开始，不然位置编码会出错。
>
> 模型参数量：**214,656**。非常小的玩具模型——4 层、4 头、64 维、vocab=100。但足够演示 KV Cache 的效果。
>
> 运行 Cell 15，性能测试。（运行 Cell 15）
>
> 看结果：
> - 生成 10 tokens：Naive=0.079s，Cache=0.015s，**加速 5.3x**
> - 生成 20 tokens：Naive=0.039s，Cache=0.034s，加速 1.1x
> - 生成 50 tokens：Naive=0.091s，Cache=0.077s，加速 1.2x
>
> 嗯，加速效果没有理论上的 50x 那么夸张。为什么？因为我们的模型太小了！214K 参数，PyTorch 的调度开销和 CUDA kernel launch 开销占了大头。对于 7B、70B 的大模型，Attention 计算才是真正的瓶颈，KV Cache 的加速效果会非常显著。
>
> 运行 Cell 16 看可视化图。（运行 Cell 16）左图是时间对比，红色朴素、绿色 Cache。右图是加速倍数柱状图，生成 10 tokens 时加速最明显（5.3x），后面稳定在 1.2x 左右。
>
> 记住：**理论加速是 O(n) vs O(n²)，实际加速取决于模型大小和硬件。** 小模型 overhead 大，大模型才能发挥 KV Cache 的真正威力。HuggingFace 的 generate 函数默认就开了 KV Cache。

👀 输出要点
- Cell 13：模型参数量 214,656
- Cell 15 性能数据：
  - 10 tokens: 5.3x 加速
  - 20 tokens: 1.1x
  - 30 tokens: 1.2x
  - 40 tokens: 1.2x
  - 50 tokens: 1.2x
- Cell 16：时间对比图 + 加速倍数柱状图

❓ 预判问题
- **Q：为什么小模型加速效果不明显？**
  A：因为 Python/PyTorch 的函数调用开销、CUDA kernel 启动开销等固定成本在小模型中占比大。大模型的 Attention 矩阵运算是真正的计算密集型操作，KV Cache 省掉的计算量远大于这些固定开销。
- **Q：`generate_with_cache` 里为什么第一步处理完整 prompt？**
  A：这就是 Prefill 阶段——首次处理 prompt 时没有缓存，需要算出所有位置的 K/V。之后的 Decode 阶段每步只处理一个新 token，用上一步的缓存。

➡️ 转场

> 性能对比完了。最后一个重要话题——KV Cache 到底吃多少显存？我们来算算真实模型的内存占用。

---

## 第六段：KV Cache 内存分析（Cell 17-18）

📍 浏览 Cell 17（Markdown 公式说明）、运行 Cell 18（真实模型内存对比）

⏱ 时间分配：7 分钟

🎯 本段目标
- 掌握 KV Cache 内存计算公式
- 直观感受不同模型的内存占用差异
- 理解 GQA 如何压缩 KV Cache 内存

🗣 讲课话术

> Cell 17 总结了公式：`KV = 2 * n_layer * batch * n_kv_head * seq_len * head_dim * dtype_bytes`。注意这里的 `n_kv_head` 而不是 `n_head`——如果模型用了 GQA，KV 头数比 Q 头数少。
>
> Cell 18 的代码里有个 TODO：`kv_size = 2 * batch_size * kv_heads * seq_len * head_dim * dtype_bytes`。这就是单层的 KV Cache 大小，乘以 n_layer 就是总量。代码还从 HuggingFace 自动拉取了真实模型的配置。
>
> 运行 Cell 18。（运行 Cell 18）这张柱状图太直观了！
>
> 看数据（batch=1, seq=2048, FP16）：
> - **Qwen2.5-0.5B：24 MB**——很小，因为只有 2 个 KV 头（GQA 激进）
> - **GPT-2 Small：72 MB**——12 层 12 头，没有 GQA
> - **Qwen2.5-7B：112 MB**——28 层但只有 4 个 KV 头
> - **DeepSeek-LLM-7B：960 MB**——30 层 32 头，没有 GQA，快 1 个 G 了！
> - **LLaMA-7B：1024 MB = 1 GB**——32 层 32 头，也没有 GQA
> - **LLaMA-13B：1600 MB = 1.56 GB**——最大的一个
>
> 有意思的对比：同样是 7B 级别的模型，**DeepSeek-LLM-7B 要 960 MB** 而 **Qwen2.5-7B 只要 112 MB**——差 8.5 倍！原因就是 GQA。Qwen2.5-7B 只有 4 个 KV 头，DeepSeek-LLM-7B 有 32 个。
>
> 实际部署中，KV Cache 内存经常是限制 batch size 和最大序列长度的瓶颈。这也是为什么 vLLM 做了 PagedAttention——像操作系统管理虚拟内存一样管理 KV Cache，避免内存碎片。

👀 输出要点
- Cell 18 表格（部分关键数据）：
  - Qwen2.5-0.5B: 24 MB, kv_heads=2
  - GPT-2 Small: 72 MB, kv_heads=12
  - Qwen2.5-7B: 112 MB, kv_heads=4
  - DeepSeek-LLM-7B: 960 MB, kv_heads=32
  - LLaMA-7B: 1024 MB, kv_heads=32
  - LLaMA-13B: 1600 MB, kv_heads=40
- Cell 18 柱状图：各模型 KV Cache 对比

❓ 预判问题
- **Q：seq_len 从 2048 增加到 128K 怎么办？**
  A：KV Cache 和 seq_len 线性相关。128K / 2048 = 62.5 倍。LLaMA-7B 从 1 GB 变成 62.5 GB——一张 A100 80GB 都不够！所以长上下文模型必须用 GQA + 量化 + 滑动窗口等优化。
- **Q：FP16 能换成 INT8 吗？**
  A：可以！KV Cache 量化是活跃的研究方向。INT8 直接把内存减半，INT4 减到四分之一。代价是精度略有下降。本章 Extra 部分提到了这个方向。
- **Q：这个 TODO 让我们补全什么？**
  A：代码已经写好了。TODO 注释是提醒大家理解公式：`2 * batch_size * kv_heads * seq_len * head_dim * dtype_bytes`。2 是 K 和 V 各一份。

➡️ 转场

> 最后我们来总结一下本章的核心知识点，再看看有哪些练习可以课后做。

---

## 第七段：总结 + 练习 + 下一步（Cell 19-22）

📍 浏览 Cell 19（本章总结）、Cell 20（下一步）、Cell 21（Extra 思考题）、Cell 22（练习空间）

⏱ 时间分配：5 分钟

🎯 本段目标
- 回顾本章核心概念
- 布置课后练习
- 预告下一章

🗣 讲课话术

> 看 Cell 19 的总结图谱。左边朴素方法：每步重算全部 Q/K/V，Attention 是 n×n 矩阵，O(n²)。右边 KV Cache：只算新 token 的 Q/K/V，Attention 是 1×n 向量，O(n)。中间的表格是经典的时间-空间权衡。
>
> 关键公式速查表大家拍个照：KV Cache 内存公式、朴素 vs Cache 计算量、加速比 = (n+1)/2。
>
> Cell 19 底部还有四道面试题——**为什么只缓存 K 和 V？MQA 和 GQA 的区别？GPT-3 的 KV Cache 要多少内存？滑动窗口怎么工作？** 这四题在大厂面试中非常高频，大家课后一定要过一遍答案。
>
> Cell 21 的 Extra 有三个进阶方向：
> 1. **实现滑动窗口 KV Cache**——只缓存最近 W 个 token，内存固定为 O(W)
> 2. **INT8 量化缓存**——用 `torch.quantize_per_tensor` 把 FP16 的 K/V 压缩到 INT8
> 3. **思考题：为什么 Flash Attention 不需要显式 KV Cache？**——提示：Flash Attention 是在训练时优化 IO，推理时仍然需要 KV Cache
>
> 下一章 Ch12 是 Agent & RAG——让 LLM 使用工具和检索信息。那里会用到 ReAct 范式、Function Calling、向量检索。LLM 推理优化是基础设施层，Agent 和 RAG 是应用层——有了快速推理，才能支撑 Agent 的多轮调用。

### 练习：Extra 思考题

**提示节奏**
- 0-2 分钟：自己思考，尤其是第 3 题"Flash Attention 为什么不需要显式 KV Cache"
- 2 分钟提示：Flash Attention 是在训练时的优化（IO-aware），它重组了 Attention 的计算顺序来减少 HBM 访问。推理时逐 token 生成仍然需要 KV Cache
- 4 分钟关键思路：Flash Attention 解决的是"一次 Attention 计算中的内存访问效率"，KV Cache 解决的是"多步生成中的重复计算"，两者是正交的优化

**常见错误**
- 混淆 Flash Attention 和 KV Cache 的优化目标
- 以为 KV Cache 只在训练时有用（实际只在推理时有用）
- 忘记 GQA 的 kv_heads < n_heads，用 n_heads 算内存会偏大

**验证标准**
- 能清晰区分 KV Cache（避免重复计算）和 Flash Attention（优化 IO）
- 能手算给定模型配置的 KV Cache 内存
- 能解释为什么 GQA 能压缩 KV Cache

👀 输出要点
- Cell 19：核心概念图谱 + 公式速查表 + 4 道面试题
- Cell 21：3 道 Extra 思考题

❓ 预判问题
- **Q：vLLM 的 PagedAttention 是怎么回事？**
  A：借鉴操作系统的虚拟内存分页。传统 KV Cache 需要连续的大块显存，容易碎片化。PagedAttention 把 Cache 按固定大小的"页"分配，页可以不连续存放，需要时动态分配和回收。这让多请求并发时显存利用率大幅提升。
- **Q：实际生产中 KV Cache 有多重要？**
  A：极其重要。所有主流推理框架（vLLM、TensorRT-LLM、Ollama、HuggingFace TGI）都默认启用 KV Cache。没有它，大模型的推理延迟和吞吐量完全不可接受。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，运行环境准备 | 0-3 |
| 5 | 问题分析：O(n²) 的计算瓶颈 | 4-5 |
| 15 | KV Cache 原理 + 可视化 | 6-10 |
| 30 | **休息 + 回顾** | -- |
| 35 | 代码实现：带 Cache 的 Attention | 11 |
| 42 | 完整 GPT + 性能对比 | 12-16 |
| 48 | KV Cache 内存分析 | 17-18 |
| 55 | 总结 + 练习 | 19-22 |
| 60 | 下课 | -- |

---

## 附录 B：关键数据快速参考

### 核心公式

$$\text{KV Cache Memory} = 2 \times L \times B \times n_{kv\_heads} \times T \times d_{head} \times \text{dtype\_bytes}$$

$$\text{朴素总计算量} = \sum_{i=1}^{n} i = \frac{n(n+1)}{2} = O(n^2)$$

$$\text{Cache 总计算量} = n \times O(1) = O(n)$$

$$\text{加速比} = \frac{n+1}{2} \approx \frac{n}{2}$$

### 演示模型配置

| 参数 | 值 |
|:---|:---|
| vocab_size | 100 |
| block_size | 128 |
| n_layer | 4 |
| n_head | 4 |
| n_embd | 64 |
| 总参数量 | 214,656 |

### 性能测试关键数值

| 生成 Token 数 | 朴素耗时 | Cache 耗时 | 加速倍数 |
|:---|:---|:---|:---|
| 10 | 0.079s | 0.015s | 5.3x |
| 20 | 0.039s | 0.034s | 1.1x |
| 30 | 0.058s | 0.049s | 1.2x |
| 40 | 0.079s | 0.064s | 1.2x |
| 50 | 0.091s | 0.077s | 1.2x |

### 理论 vs 实际加速说明

序列长度 100 时理论加速 50.5x，实测小模型仅 1.2-5.3x。原因：模型太小（214K 参数），Python/CUDA 调度开销占主导。大模型（7B+）实际加速效果显著。

### KV Cache 内存对比（batch=1, seq=2048, FP16）

| 模型 | KV Cache | KV 头数 | 说明 |
|:---|:---|:---|:---|
| Qwen2.5-0.5B | 24 MB | 2 | GQA 激进 |
| GPT-2 Small | 72 MB | 12 | 无 GQA |
| Qwen2.5-7B | 112 MB | 4 | GQA |
| DeepSeek-LLM-7B | 960 MB | 32 | 无 GQA |
| LLaMA-7B | 1024 MB | 32 | 无 GQA |
| LLaMA-13B | 1600 MB | 40 | 无 GQA |

### GPT-3 / LLaMA-2-70B 理论 KV Cache

| 模型 | 配置 | KV Cache (FP16, batch=1) |
|:---|:---|:---|
| GPT-3 (175B) | 96 层, 96 头, seq=2048 | 9.66 GB |
| LLaMA-2-70B (GQA) | 80 层, 8 KV 头, seq=4096 | 1.34 GB |

---

## 附录 C：应急预案

### 场景 1：环境问题

**症状：** `ModuleNotFoundError: No module named 'torch'`

**应对：**
1. `pip install torch`
2. 重启 Kernel 后重新运行

### 场景 2：transformers 库未安装

**症状：** Cell 18 报错 `ModuleNotFoundError: No module named 'transformers'`

**应对：**
1. `pip install transformers`
2. 如果 HuggingFace 无法访问（网络问题），代码有 `fallback_specs` 兜底配置，GPT-2 和 LLaMA 的数据仍会显示
3. 可跳过 Cell 18，直接用 Cell 8 的理论数值讲解内存分析

### 场景 3：中文字体不显示

**症状：** Matplotlib 图表中中文显示为方框

**应对：**
1. Cell 3 已设置 `plt.rcParams["font.sans-serif"]` 为多个中文字体候选
2. Windows 一般有 Microsoft YaHei，macOS 有 Arial Unicode MS
3. 如仍不显示，安装 SimHei 或 Noto Sans CJK

### 场景 4：性能测试数值差异大

**症状：** Cell 15 的加速倍数和教案中的数值不一致

**应对：**
1. 这是正常的——性能数据受 GPU 型号、系统负载、PyTorch 版本影响
2. 关键是看**趋势**：Cache 方法应该不慢于朴素方法
3. 小模型加速不明显是预期行为，可口头解释原因（开销占比大）
4. 如 CPU 上跑太慢，可减少 `lengths` 列表中的值

### 场景 5：HuggingFace 模型配置下载失败

**症状：** Cell 18 多个模型 `load_spec_from_hf` 报错

**应对：**
1. 代码中 `fallback_specs` 会自动兜底 GPT-2 系列和 LLaMA 系列
2. 其他模型（Qwen、DeepSeek、ERNIE）需要网络访问，无法下载时会自动跳过
3. 手动添加 fallback：在 `fallback_specs` 字典中补充缺失模型的配置
4. 核心教学点（内存公式、GQA 压缩效果）不受影响

### 场景 6：CUDA 内存不足

**症状：** Cell 15 性能测试时 `RuntimeError: CUDA out of memory`

**应对：**
1. 模型很小（214K 参数），一般不会 OOM
2. 如果其他程序占用了 GPU 显存，关闭不需要的程序
3. 切换到 CPU：修改 Cell 3 的 device 为 `cpu`，性能测试仍可运行（数值会不同但趋势一致）